# Notebook06 fixed-probe evaluation definitions

Reusable evaluation definitions; importing this notebook does not read images or train. Inputs: complete paired training evidence, immutable initialization/final encoder states, fixed 10,100 labeled probe images and 2,020 development images. Outputs: resumable epoch20 probes, reconstructed predictions, initialization-adjusted paired seed contrasts, timing/memory summaries and a report. The official test set remains reserved. Expected evaluation time is reserved by the three-hour workflow. Checkpoints: `checkpoints/<run_id>/stages/learning/probes/`. Resume with Run All in the unchanged Notebook06 and run ID; deadline never resets.

In [ ]:
"""Notebook06 fixed epoch20 probes and paired exploratory evidence."""
from pathlib import Path
import copy, hashlib, json, math, sys, time
import numpy as np
import torch

ROOT=Path(__file__).resolve().parents[2]
if 'fmm06_runner' not in sys.modules:
    import types
    _path=ROOT/'experiments/06_math/runner.ipynb'
    _module=types.ModuleType('fmm06_runner');_module.__file__=str(_path);sys.modules['fmm06_runner']=_module
    for _cell in json.loads(_path.read_text())['cells']:
        if _cell['cell_type']=='code' and 'ahr-export' in _cell.get('metadata',{}).get('tags',[]):
            exec(compile(''.join(_cell['source']),str(_path),'exec'),_module.__dict__)
RUNNER=sys.modules['fmm06_runner']
_BASE=RUNNER.load_tagged(ROOT,'experiments/05_scaling/evaluation.ipynb','fmm06_inherited_evaluation')
for _name,_value in list(_BASE.__dict__.items()):
    if not _name.startswith('__') and _name not in globals():globals()[_name]=_value
build_model=RUNNER.build_model


def _evaluation_export_hash(root):
    return _digest([''.join(c['source']) for c in _read(Path(root)/'experiments/06_math/evaluation.ipynb')['cells']
        if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])])

def _verify_plan_sources(root,plan):
    actual=RUNNER.stage_definition_hashes(root)
    if plan['source_hashes']!=actual:raise ValueError('Notebook06 registered definitions changed')
    if plan.get('consumer_notebook_sha256'):
        code=[''.join(c['source']) for c in _read(Path(root)/plan['resume_notebook'])['cells'] if c['cell_type']=='code']
        if _digest(code)!=plan['consumer_notebook_sha256']:raise ValueError('Registered main notebook code changed')
    return actual

def _stage_jobs(plan,training):
    if training.get('status')!='COMPLETE' or not training.get('equal_updates') or not training.get('paired_traces_within_seed') or not training.get('no_failed_attempts'):
        raise ValueError('Only clean, fully paired, equal-update training may enter quality evaluation')
    if set(training['arms'])!={a['id'] for a in plan['arms']}:raise ValueError('Training arm identities differ')
    jobs=[]
    for arm in plan['arms']:
        if arm['config']['training']['updates']!=plan['updates'] or arm['config']['probe']['epochs']!=20:
            raise ValueError('Schedule/probe budget differs')
        if arm['config']['evaluation'].get('heldout_enabled',False):raise ValueError('Official test remains reserved')
        record=training['arms'][arm['id']]
        if record['successful_updates']!=plan['updates'] or record['attempted_updates']!=plan['updates'] or record['failures']:
            raise ValueError('Incomplete or numerically failed arm')
        jobs.append({'model_id':arm['id']+'-initial','arm':arm,'updates':0,
            'control':'initialization','state':record['initial_teacher'],'training_cost':None})
        for milestone in plan['milestones']:
            state=record['milestones'][str(milestone)]
            if state['successful_updates']!=milestone:raise ValueError('Milestone update differs')
            jobs.append({'model_id':arm['id']+'-u'+str(milestone),'arm':arm,'updates':milestone,
                'control':'trained','state':state,'training_cost':state})
    return jobs

def _seed_statistics(rows,value_key='delta_percentage_points'):
    n=len(rows)
    if n not in (2,3) or len({r['seed'] for r in rows})!=n:raise ValueError('Two or three distinct paired training seeds required')
    values=[float(r[value_key]) for r in rows]
    if not all(math.isfinite(x) for x in values):raise ValueError('Nonfinite seed effect')
    mean=sum(values)/n;sd=math.sqrt(sum((x-mean)**2 for x in values)/(n-1))
    quantile=math.tan(math.pi*.475) if n==2 else math.sqrt(2*.95**2/(1-.95**2))
    half=quantile*sd/math.sqrt(n)
    return {'n_training_seeds':n,'mean_percentage_points':mean,'sd_percentage_points':sd,
        'approximate_t_ci95_percentage_points':[mean-half,mean+half],
        'range_percentage_points':[min(values),max(values)],'positive_seeds':sum(x>0 for x in values),
        'paired_seed_deltas':[{'seed':int(row['seed']),'delta_percentage_points':float(row[value_key])} for row in rows],
        'uncertainty':'Paired training seeds are the replication units; tiny-sample t intervals are exploratory, conditional on reused development membership and not multiplicity-adjusted.'}

def analyze_stage_evidence(evaluations):
    index={(v['variant'],int(v['seed']),int(v['updates'])):v for v in evaluations.values()}
    seeds=sorted({k[1] for k in index});variants=sorted({k[0] for k in index});steps=sorted({k[2] for k in index});final=max(steps)
    if len(index)!=len(seeds)*len(variants)*len(steps):raise ValueError('Incomplete crossed seed/variant/update evidence')
    contrasts={};rows=[]
    for variant in variants:
        for step in steps[1:]:
            contrasts[variant+'_learning_u'+str(step)]=_seed_statistics([{'seed':seed,'delta_percentage_points':
                _paired_delta(index[(variant,seed,step)],index[(variant,seed,0)])}for seed in seeds])
    for comparator in ('stable','vit'):
        for step in steps:
            contrasts['centered_tree_minus_'+comparator+'_u'+str(step)]=_seed_statistics([{'seed':seed,'delta_percentage_points':
                _paired_delta(index[('centered_tree',seed,step)],index[(comparator,seed,step)])}for seed in seeds])
            if step:
                contrasts['centered_tree_minus_'+comparator+'_learning_gain_u'+str(step)]=_seed_statistics([{'seed':seed,'delta_percentage_points':
                    _paired_delta(index[('centered_tree',seed,step)],index[('centered_tree',seed,0)])-
                    _paired_delta(index[(comparator,seed,step)],index[(comparator,seed,0)])}for seed in seeds])
    for variant in variants:
        for seed in seeds:
            initial=_score(index[(variant,seed,0)])
            for step in steps:
                saved=index[(variant,seed,step)];cost=saved.get('training_cost')or{};perf=cost.get('performance')or{}
                peak=perf.get('peak_cuda_allocated_bytes')
                rows.append({'model_id':saved['model_id'],'variant':variant,'kind':saved['kind'],'seed':seed,'updates':step,
                    'top1_percent':_score(saved),'initial_top1_percent':initial,'gain_percentage_points':_score(saved)-initial,
                    'image_exposures':int(cost.get('image_exposures',0)),
                    'cumulative_training_step_seconds':cost.get('cumulative_step_seconds')if step else 0.,
                    'median_training_step_seconds':(perf.get('complete_jepa_step_seconds')or{}).get('median'),
                    'peak_training_GiB':float(peak)/1024**3 if peak is not None else None})
    sanity={str(seed):{'finite':bool(index[('centered_tree',seed,final)]['representation_diagnostics']['finite_train_and_development']),
        'not_exactly_constant':bool(index[('centered_tree',seed,final)]['representation_diagnostics']['development_representation_not_exactly_constant'] and index[('centered_tree',seed,final)]['representation_diagnostics']['training_representation_not_exactly_constant'])}for seed in seeds}
    learning=contrasts['centered_tree_learning_u'+str(final)];lower=learning['approximate_t_ci95_percentage_points'][0]
    passed=lower>0 and all(x['finite']and x['not_exactly_constant']for x in sanity.values())
    gate={'passed':passed,'status':'PASS'if passed else'LEARNING_UNESTABLISHED','representation_checks':sanity,
        'lower_t_bound_percentage_points':lower,'reason':'Exploratory own-initialization learning assessment; no stage selection or automatic continuation.',
        'caution':'A three-hour pilot cannot guarantee a statistically resolved superiority result.'}
    return {'contrasts':contrasts,'chart_rows':rows,'learning_gate':gate}

def write_stage_figures(artifact,result,summary):
    """Portable figures backed by exactly the saved row/contrast data."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    variants=['vit','stable','centered_tree'];labels=['ViT','Stable semantic','Centered tree'];colors=['#59636E','#4B77A8','#CE8436']
    final=result['training_design']['updates'];steps=[0,final]
    fig,axes=plt.subplots(1,2,figsize=(12,4.7),layout='constrained')
    for i,variant in enumerate(variants):
        rows=[r for r in result['chart_rows'] if r['variant']==variant]
        for seed in sorted({r['seed']for r in rows}):
            line=sorted([r for r in rows if r['seed']==seed],key=lambda r:r['updates'])
            axes[0].plot([r['updates']for r in line],[r['top1_percent']for r in line],color=colors[i],alpha=.3,linewidth=1)
        means=[float(np.mean([r['top1_percent']for r in rows if r['updates']==step]))for step in steps]
        axes[0].plot(steps,means,color=colors[i],marker=['o','s','^'][i],linewidth=2,label=labels[i])
        stats=result['contrasts'][variant+'_learning_u'+str(final)];mean=stats['mean_percentage_points'];lo,hi=stats['approximate_t_ci95_percentage_points']
        axes[1].errorbar(mean,i,xerr=[[mean-lo],[hi-mean]],fmt=['o','s','^'][i],color=colors[i],capsize=4,markersize=7)
    axes[0].set(title='Development accuracy before and after training',xlabel='Successful training updates',ylabel='Top1 accuracy (%)',xticks=steps)
    axes[0].set_ylim(bottom=0);axes[0].legend(frameon=False)
    axes[1].axvline(0,color='#59636E',linewidth=1);axes[1].set(title='Learning over each initialization',xlabel='Gain (percentage points); exploratory 95% t interval',yticks=range(3),yticklabels=labels)
    for axis in axes:axis.spines[['top','right']].set_visible(False);axis.grid(axis='x',alpha=.15)
    fig.suptitle('Food101 development: fixed epoch20 probes; paired training seeds',fontsize=12)
    quality=Path(artifact)/'stage_quality.png';fig.savefig(quality,dpi=160);plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(12,4.4),layout='constrained')
    for axis,key,title,unit in ((axes[0],'step_seconds','Training step time','Seconds'),(axes[1],'peak_GiB','Peak allocated GPU memory','GiB')):
        values=[r[key]for r in summary]
        if any(v is None for v in values):axis.text(.5,.5,'UNKNOWN: incomplete measurements',ha='center',transform=axis.transAxes)
        else:
            axis.bar(range(3),values,color=colors,edgecolor='#333333',linewidth=.5)
            for i,value in enumerate(values):axis.annotate(f'{value:.3f}',(i,value),xytext=(0,4),textcoords='offset points',ha='center')
            axis.set_ylim(0,max(values)*1.18)
        axis.set(title=title,ylabel=unit,xticks=range(3),xticklabels=labels);axis.spines[['top','right']].set_visible(False)
    fig.suptitle('Means of per-seed measurements; same update budget, not matched compute',fontsize=12)
    compute=Path(artifact)/'stage_compute.png';fig.savefig(compute,dpi=160);plt.close(fig)
    records={name:{'path':str(path.resolve()),'sha256':digest_file(path)} for name,path in [('quality',quality),('compute',compute)]}
    records['visual_inspection']='Pending human/agent inspection after real experiment completion; no quality results fabricated during build.'
    atomic_json(Path(artifact)/'stage_figure_manifest.json',records)
    return records

def write_stage_report(artifact_dir,result):
    artifact=Path(artifact_dir);final=result['training_design']['updates'];rows=[r for r in result['chart_rows']if r['updates']==final]
    variants=['vit','stable','centered_tree'];summary=[]
    for variant in variants:
        group=[r for r in rows if r['variant']==variant]
        mean=lambda name:float(np.mean([r[name] for r in group]))if all(r[name]is not None for r in group)else None
        summary.append({'variant':variant,'top1_percent':mean('top1_percent'),'gain_percentage_points':mean('gain_percentage_points'),
            'step_seconds':mean('median_training_step_seconds'),'peak_GiB':mean('peak_training_GiB')})
    indexed={r['variant']:r for r in summary};ratios={}
    for comparator in ('stable','vit'):
        ratios[comparator]={}
        for field in ('step_seconds','peak_GiB'):
            a=indexed['centered_tree'][field];b=indexed[comparator][field]
            ratios[comparator][field+'_candidate_over_comparator']=a/b if a is not None and b is not None and b>0 else None
        speed=ratios[comparator]['step_seconds_candidate_over_comparator']
        ratios[comparator]['descriptive_cost_signal']='AT_LEAST_10_PERCENT_FASTER' if speed is not None and speed<=.9 else 'NO_10_PERCENT_SPEEDUP_OBSERVED'
    atomic_json(artifact/'stage_cost_comparisons.json',ratios)
    figures=write_stage_figures(artifact,result,summary)
    lines=['# Notebook06 mathematical changes: three-hour pilot' ,'',
        'Status: COMPLETE. All quality scores use fixed epoch20 linear probes and the same held-out development identities. Official test: NOT_RUN.',
        '',f"Training budget: {final} successful updates per arm; {len(result['evaluations'])//6} paired seeds; 128 images per update. All variants share the full schedule and image/mask stream.",
        '', '| Variant | Development top1 (%) | Gain over initialization (pp) | Median step (s) | Peak GPU memory (GiB) |',
        '|---|---:|---:|---:|---:|']
    for row in summary:
        show=lambda key:'UNKNOWN'if row[key]is None else f"{row[key]:.4f}"
        lines.append('| '+row['variant']+' | '+' | '.join(show(k)for k in('top1_percent','gain_percentage_points','step_seconds','peak_GiB'))+' |')
    lines+=['','The primary comparison is centered_tree versus stable semantic. ViT provides an external reference. Equal updates are not equal compute. Centering, tree evaluation and a training-only selected per-layer feature allocation are bundled in the trained candidate; operator diagnostics provide separate ablations.',
        '', '| Paired contrast | Mean (pp) | Exploratory 95% t interval (pp) |','|---|---:|---:|']
    for name,value in result['contrasts'].items():
        if name.endswith('u'+str(final)):
            lo,hi=value['approximate_t_ci95_percentage_points'];lines.append(f"| {name} | {value['mean_percentage_points']:.4f} | [{lo:.4f}, {hi:.4f}] |")
    lines+=['','Uncertainty uses training seeds, not individual development images, as replication units. Two or three seeds give fragile intervals; comparisons are exploratory and not multiplicity-adjusted. A confidence interval containing zero establishes neither equivalence nor absence of a difference.',
        '', 'Timing includes ordinary training-step data preparation and GPU compute, excluding probe/checkpoint overhead. Peak memory is allocated CUDA memory, not total device reservation. Initial and final probe feature caches, reconstructed predictions, and immutable full-state checkpoints support every score.',
        '', 'Resume: Run All in Notebook06 with the same run ID and unchanged sources. Completed results are reused. An exhausted deadline never extends automatically.']
    lines+=['','![Development quality and own-initialization learning](stage_quality.png)',
        '','![Training step time and peak allocated GPU memory](stage_compute.png)',
        '', 'Cost ratios (candidate / comparator): '+json.dumps(ratios,sort_keys=True)+'. These are descriptive measurements, not a universal speed guarantee.']
    path=artifact/'stage_results.md';path.write_text('\n'.join(lines)+'\n')
    atomic_json(artifact/'stage_summary_table.json',summary)
    return {'report':str(path.resolve()),'summary_table':str((artifact/'stage_summary_table.json').resolve()),'figures':figures,'cost_comparisons':ratios}

def _evaluate_stage_locked(root, artifact, plan, bundle, training, deadline):
    sources = _verify_plan_sources(root, plan)
    membership = _fixed_membership(root, bundle)
    jobs = _stage_jobs(plan, training)
    stage_role = plan.get("stage_role", plan["stage_id"])
    if stage_role not in STAGE_ROLES: raise ValueError("Unknown stage_role")
    if membership["pretraining_count"] != (65600 if stage_role == "data_volume" else 10100):
        raise ValueError("Pretraining count differs from the registered stage role")
    if int(plan["arms"][0]["config"]["image_size"]) != (256 if stage_role == "resolution" else 128):
        raise ValueError("Stage image resolution differs from declared role")
    identity = {"schema": "notebook05-stage-evaluation/v1", "plan": _canonical(plan),
        "source_hashes": sources, "evaluation_export_hash": _evaluation_export_hash(root),
        "dataset_fingerprint": bundle["dataset"].fingerprint, "membership": membership,
        "jobs": _canonical(jobs), "probe_selection": "fixed_final_epoch20", "official_test": "NOT_RUN"}
    identity_hash = _digest(identity)
    manifest_path = artifact/"stage_evaluation_manifest.json"
    if manifest_path.exists():
        manifest = _read(manifest_path)
        if manifest["identity_hash"] != identity_hash: raise ValueError("Stage evaluation identity changed; refusing stale reuse")
    else:
        manifest = {"identity_hash": identity_hash, "identity": identity, "completed": {}, "started_unix": time.time()}
        atomic_json(manifest_path, manifest)
    evaluations = {}
    for job in jobs:
        model_id, arm = job["model_id"], job["arm"]
        config = copy.deepcopy(arm["config"])
        summary_path = artifact/"evaluation"/(model_id+".json")
        _verified_record(job["state"])
        if model_id in manifest["completed"]:
            evaluations[model_id] = _load_saved_job(summary_path, manifest["completed"][model_id], identity_hash, job, membership)
            continue
        _deadline_check(deadline, 5)
        configure_environment(config)
        atomic_json(artifact/"evaluation_status.json", {"status":"RUNNING", "model_id":model_id,
            "completed_models":len(evaluations), "total_models":len(jobs), "updated_unix":time.time()})
        model = encoder = None
        try:
            _release_semantic(config["device"])
            model = build_model(config, arm["kind"]).cpu()
            state = torch.load(job["state"]["path"], map_location="cpu", weights_only=job["updates"] == 0)
            if job["updates"] == 0:
                model.target.load_state_dict(state)
            else:
                if int(state["state"]["successful_updates"]) != job["updates"] or state["config"] != arm["config"]:
                    raise ValueError("Checkpoint milestone/config differs from the registered 1000-update schedule")
                model.load_state_dict(state["model"])
            del state
            encoder = model.target
            model = None
            common = {"dataset_revision":bundle["dataset"].fingerprint, "checkpoint_hash":encoder_state_hash(encoder),
                "encoder_kind":arm["kind"], "preprocessing":preprocessing_descriptor(config),
                "pooling":"mean_final_layernorm_patches",
                "normalization":"encoder final affine LayerNorm then patch mean; train-only per-feature probe standardization"}
            caches, extraction = {}, []
            for split, ids in (("train", membership["probe_train_ids"]), ("development", membership["development_ids"])):
                _deadline_check(deadline, 2)
                dataset = _DeadlineDataset(EvaluationDataset(bundle["dataset"], ids, config), deadline)
                print(stage_role, model_id, split, len(ids), "fixed labeled feature IDs", flush=True)
                cache = extract_features(encoder, dataset, artifact/"features"/model_id,
                    {**common,"sample_ids":ids}, batch_size=int(config["evaluation"]["batch_size"]), device=config["device"])
                caches[split] = cache
                extraction.append({"split":split, "cache_reused":cache["reused"], **cache["performance"]})
            encoder.cpu()
            _release_semantic(config["device"])
            diagnostics = _cached_representation_diagnostics(caches["train"], caches["development"], bundle.get("diagnostic_ids", []))
            result = _fit_fixed_probe(caches["train"], caches["development"],
                root/"checkpoints"/plan["run_id"]/"stages"/plan["stage_id"]/"probes", config, model_id, encoder, deadline, plan)
            saved = {"identity_hash":identity_hash, "model_id":model_id, "arm_id":arm["id"], "kind":arm["kind"], "variant":arm["variant"],
                "seed":int(arm["seed"]), "stage_role":stage_role, "updates":job["updates"], "control":job["control"],
                "source_encoder":job["state"], "feature_caches":{name:_state_record(cache["path"]) for name,cache in caches.items()},
                "probe":result, "representation_diagnostics":diagnostics, "feature_extraction":extraction,
                "training_cost":job["training_cost"], "heldout_status":"NOT_RUN"}
            write_csv(artifact/"predictions"/(model_id+".csv"), result["predictions"])
            atomic_json(summary_path, saved)
            manifest["completed"][model_id] = _state_record(summary_path)
            atomic_json(manifest_path, manifest)
            evaluations[model_id] = saved
            print(stage_role, model_id, "fixed epoch20 top1", 100*result["summary"]["metrics"]["top1"], flush=True)
        finally:
            model = encoder = None
            _release_semantic(config["device"])
    _deadline_check(deadline)
    if len(evaluations) != len(jobs): raise AssertionError("Every registered initial/final probe is required")
    probe_inits = {x["probe"]["summary"]["initialization_hash"] for x in evaluations.values()}
    probe_budgets = {_digest(x["probe"]["summary"]["budget"]) for x in evaluations.values()}
    probe_orders = {_digest([(r["epoch"],r["split"],r["permutation_hash"]) for r in x["probe"]["metrics"]]) for x in evaluations.values()}
    if len(probe_inits) != 1 or len(probe_budgets) != 1 or len(probe_orders) != 1:
        raise AssertionError("Probe initialization/budget/order is not identical across stage comparisons")
    scientific = analyze_stage_evidence(evaluations)
    result = {"schema":"notebook05-stage-evaluation/v1", "status":"COMPLETE", "run_id":plan["run_id"],
        "stage_id":plan["stage_id"], "stage_role":stage_role, "identity_hash":identity_hash,
        "plan_hash":_digest(plan), "dataset_fingerprint":bundle["dataset"].fingerprint,
        "membership":membership, "evaluations":evaluations, "milestones":[0]+list(plan["milestones"]),
        "schedule_note":"One common complete schedule frozen after discarded timing calibration; initialized and final encoders use fixed epoch20 probes.",
        "image_size":int(plan["arms"][0]["config"]["image_size"]), "pretraining_images":len(bundle["train_ids"]),
        "training_design":{"updates":int(plan["updates"]),"effective_batch":plan["arms"][0]["config"]["training"]["microbatch"]*plan["arms"][0]["config"]["training"]["accumulation_steps"],
            "lr":plan["arms"][0]["config"]["training"]["lr"],"ema_start":.99,
            "weight_decay":plan["arms"][0]["config"]["training"]["weight_decay"],
            "warmup_fraction":plan["arms"][0]["config"]["training"]["warmup_fraction"],
            "patch_size":plan["arms"][0]["config"]["model"]["patch_size"],
            "model":copy.deepcopy(plan["arms"][0]["config"]["model"]), "dtype":plan["arms"][0]["config"].get("dtype","float32") },
        "probe_selection":"fixed_final_epoch20", "labels_in_self_supervised_loss":False,
        "official_test":"NOT_RUN", "started_unix":manifest["started_unix"], "completed_unix":time.time(),
        "learning_gate":scientific["learning_gate"], "contrasts":scientific["contrasts"], "chart_rows":scientific["chart_rows"],
        "report_scope":"Two or three paired seeds, exploratory and conditional on reused development membership; initialization-adjusted gains are reported alongside final accuracy."}
    atomic_json(artifact/"stage_evaluation.json", result)
    result["presentation"] = write_stage_report(artifact, result)
    atomic_json(artifact/"stage_evaluation.json", result)
    atomic_json(artifact/"evaluation_status.json", {"status":"COMPLETE", "completed_models":len(jobs), "learning_gate":result["learning_gate"], "updated_unix":time.time()})
    return result

for _name in ('build_model','_evaluation_export_hash','_verify_plan_sources','_stage_jobs','_seed_statistics',
              'analyze_stage_evidence','write_stage_report','_evaluate_stage_locked'):
    setattr(_BASE,_name,globals()[_name])
_BASE.RESUME_NOTEBOOK=RUNNER.RESUME_NOTEBOOK
_BASE.RESUME_CELL=RUNNER.RESUME_CELL
evaluate_stage=_BASE.evaluate_stage
